In [1]:
# =============================================================================
# CELL 1: ALL CONFIGURATION, ASSUMPTIONS, BASELINES, INPUTS
# =============================================================================

# --- Granularity: 'q' = quarterly, 'm' = monthly, 'w' = weekly ---
granularity = 'w'

# --- Date Range (inclusive) ---
START_DATE = '2026-01-01'
END_DATE = None  # None = auto-detect from today's date

# --- Query Control ---
run_every_query = True  # True = run SQL queries; False = use cached pickles

# --- Date Column per Granularity ---
# Q/M use book_date; W uses app_date (application_received_dtm)
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

# --- LOBs to Process ---
LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

# --- Rollup Groups (weighted-average aggregation of individual LOB results) ---
ROLLUP_GROUPS = {
    'Franchise Independent': ['AN', 'FLD', 'FRN', 'STG'],
    'nonKMX': ['AN', 'FRN', 'STG', 'FLD', 'ENT'],
    'POS': ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX'],
}

# --- Baselines (per individual LOB) ---
BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58, 'apr': 0.25}, #changed from 0.58
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25}, #changed from 0.55
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235}, #changed from 0.55
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55, 'apr': 0.235},  
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

# --- Model Parameters ---
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# --- DLA (Dealer Level Adjustment) Current Quarter ---
DLA_CURRENT_QUARTER = '2026 Q1'

# --- Excluded Vintages (per LOB) ---
# EXCLUDED_VINTAGES = {
#     'KMX': {'2022 M02', '2023 M11', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09',
#             '2023-44', '2023-45', '2023-46', '2023-47', '2023-48'},
#     'AN':  {'2023-14', '2023-15'},
#     'FRN': {'2023-14', '2023-15'},
#     'STG': {'2023-14', '2023-15'},
#     'FLD': {'2023-14', '2023-15'},
#     'ENT': {'2023-14', '2023-15'},
# }
EXCLUDED_VINTAGES = {}

In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# --- Derived values (do not modify) ---
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: w
Date column: app_date
Period range: 2025-12-28/2026-01-03 to 2026-04-05/2026-04-11
SQL min_date: '2026-01-01'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings (e.g. '2025 Q1', '2025 M01')."""
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [4]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']

    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        mask_soft = ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag
        mask_hard = ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag
        ula_df.loc[mask_soft, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.97 + 0.15 * ula_df.cd_perc_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[mask_hard, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.954 + 0.346 * ula_df.cd_perc_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================

run_query = False

# --- Model Scores ---
if run_query or run_every_query:
    all_original_model_scores = run_sql('postmodern_ms_query.txt', sub_list=[('{min_book_date}', min_date_sql)])
    store_pickle(all_original_model_scores, 'all_original_model_scores_pickle')
else:
    all_original_model_scores = get_pickle('all_original_model_scores_pickle')

print(f"Model scores fetched: {len(all_original_model_scores):,} records")

# --- ULA, DLA, New Recovery ---
if run_query or run_every_query:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = run_sql('vintage_level_ula_query.txt', sub_list=[('{min_book_date}', min_date_sql)], connection=conn)
        print('ULA query finished')

        dla_df = run_sql('new_dll_query.txt', connection=conn)
        print('DLA query finished')

        new_recovery = run_sql('new_recovery_queryt.txt', connection=conn)
        print('New recovery query finished')

    store_pickle((ula_df_total, dla_df, new_recovery), '(ula_df_total, rra_df_total, dla_df)_unrefined_pickle')
else:
    ula_df_total, dla_df, new_recovery = get_pickle('(ula_df_total, rra_df_total, dla_df)_unrefined_pickle')

print(f"ULA records: {len(ula_df_total):,}")

Model scores fetched: 46,339 records
ULA query finished
DLA query finished
New recovery query finished
ULA records: 549,228


In [6]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT + DATE FILTERING
# =============================================================================

# Filter out Core LOB
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])
all_original_model_scores['app_date'] = pd.to_datetime(all_original_model_scores['app_date'])
all_original_model_scores['book_date'] = pd.to_datetime(all_original_model_scores['book_date'])

# Assign period columns using pd.to_period
for df in [ula_df_total, new_recovery, all_original_model_scores]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter all DataFrames to the configured date range
for df in [ula_df_total, new_recovery, all_original_model_scores]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str for compatibility
for df in [ula_df_total, new_recovery, all_original_model_scores]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

# String version of date_col for flag comparisons
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

# Translate MTN 4.1 model scores to the normal scale before aggregation
is_mtn41 = all_original_model_scores.mtn_model == 4.1
all_original_model_scores.loc[is_mtn41, 'model_score'] = 142 + (all_original_model_scores.loc[is_mtn41, 'model_score'] - 142) * 1.5

# Aggregate model scores by period + LOB
ms_df = all_original_model_scores.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'model_score', include_groups=False
).reset_index()
ms_df['period'] = format_vintage(ms_df['period'])

print(f"Model scores aggregated: {len(ms_df)} period-LOB combinations")
print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")

Model scores aggregated: 98 period-LOB combinations
Periods in data: 14
Period range: 2025-12-28/2026-01-03 to 2026-03-29/2026-04-04


In [7]:
# =============================================================================
# CELL 7: FLAG CREATION AND DATA REFINEMENT
# =============================================================================

date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={"valid_vintage": "book_vintage"})
ula_df_total['book_vintage'] = ula_df_total['book_vintage'].replace(DLA_CURRENT_QUARTER, "current")
ula_df_total = pd.merge(ula_df_total, dla_df, how="left", on=['dealer_number', 'book_vintage'])
ula_df_total['book_vintage'] = ula_df_total['book_vintage'].replace("current", DLA_CURRENT_QUARTER)
ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag (derived from ULA job_company) ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- ULA NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.employment = ula_df_total.employment.fillna('not seasonal or waiter')
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total.employment == 'seasonal'
ula_df_total['waiter_employment_flag'] = ula_df_total.employment == 'waiter'
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type == 'softpull'
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate employment records ---
ula_df_total = ula_df_total.drop_duplicates().copy()
ula_df_total['employment_type_code'] = ula_df_total.seasonal_employment_flag * 1 + ula_df_total.waiter_employment_flag * 8
combined_employment_code_df = ula_df_total.groupby('account_number').employment_type_code.sum().reset_index()
ula_df_total = ula_df_total.drop(columns='employment_type_code').merge(combined_employment_code_df, on='account_number')
ula_df_total.seasonal_employment_flag = (ula_df_total.employment_type_code == 1) | (ula_df_total.employment_type_code == 9)
ula_df_total.waiter_employment_flag = (ula_df_total.employment_type_code == 8) | (ula_df_total.employment_type_code == 9)
ula_df_total = ula_df_total.drop(columns='employment').drop_duplicates()

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# Convert period to string for vintage-based lookups
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

store_pickle(ula_df_total, '(ula_df_total, rra_df_total)_refined_pickle')

print(f"ULA after refinement: {len(ula_df_total):,}")

ULA after refinement: 53,850


In [8]:
# =============================================================================
# CELL 8: RAGU SCORE COMPUTATION - INDIVIDUAL LOBs ONLY
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config, leave_out='None'):
    """Core RAGU Score calculation for a single vintage and individual LOB."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17 / 0.65 if lob == 'KMX' else 17
    apr_mult = 0.7 / 0.65 if lob == 'KMX' else 0.7

    ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()

    if len(ula_df) == 0:
        return None

    if lob == 'KMX':
        ula_df = get_ula_multiplier_kmx(ula_df, leave_out=leave_out)
    else:
        ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='inner').drop_duplicates(subset='account_number', keep='first')
    bb_populated_df = mix_df.dropna(subset=['bbvalue', 'recovery_multiplier'])

    if len(bb_populated_df) == 0:
        return None

    bb_populated_df['ltv'] = bb_populated_df.amt_financed / bb_populated_df.bbvalue
    bb_populated_df['recovery_unadjusted_multiplier'] = bb_populated_df['recovery_multiplier'].copy()

    grouped_mix_df = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


# --- Main RAGU Loop: Individual LOBs x Periods ---
all_vintages = sorted(ula_df_total['vintage'].unique())
results = []

for lob in LOBS:
    excluded = EXCLUDED_VINTAGES.get(lob, set())
    baseline_config = BASELINES[lob]

    for vintage in all_vintages:
        if vintage in excluded:
            continue
        try:
            result = get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config)
            if result is not None:
                results.append(result)
                rec_mult = result['recovery_unadjusted_multiplier'].values[0]
                baseline_rec = baseline_config['new_recovery_unadjusted']
                # print(f"  {vintage} {lob}: recovery_mult={rec_mult:.4f}  baseline={baseline_rec}  ratio={rec_mult/baseline_rec:.4f}")
        except Exception as e:
            print(f"Error: {vintage} {lob}: {e}")

    print(f"{lob} complete")

all_df = pd.concat(results, ignore_index=False)
all_df = all_df.reset_index()
print(f"\nTotal results: {len(all_df)} rows across {all_df.vintage.nunique()} vintages and {all_df.lob.nunique()} LOBs")

AN complete
FRN complete
STG complete
FLD complete
ENT complete
KMX complete

Total results: 84 rows across 14 vintages and 6 LOBs


In [9]:
# =============================================================================
# CELL 8b: ROLLUP AGGREGATION (POS, nonKMX)
# =============================================================================

rollup_metrics = [
    'ms_original', 'gross_loss_impact', 'recovery_impact',
    'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
]

for group_name, group_lobs in ROLLUP_GROUPS.items():
    group_df = all_df[all_df.lob.isin(group_lobs)].copy()
    group_df = group_df.rename(columns={'amt_financed_x': 'amt_financed'})
    rollup = group_df.groupby('vintage').apply(
        weighted_average_and_sum, rollup_metrics, include_groups=False
    ).reset_index()
    rollup['lob'] = group_name
    rollup = rollup.rename(columns={'amt_financed': 'amt_financed_x'})
    all_df = pd.concat([all_df, rollup], ignore_index=True)

print(f"After rollups: {len(all_df)} rows across {all_df.lob.nunique()} groups")
print(f"Groups: {sorted(all_df.lob.unique())}")

After rollups: 126 rows across 9 groups
Groups: ['AN', 'ENT', 'FLD', 'FRN', 'Franchise Independent', 'KMX', 'POS', 'STG', 'nonKMX']


In [10]:
# =============================================================================
# CELL 9: UNIFIED EXCEL EXPORT
# =============================================================================

METRIC_ROWS = [
    ('Model Score',       'ms_original'),
    ('Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',   'recovery_impact'),
    ('LTV Impact',        'ltv_impact'),
    ('APR Impact',        'apr_impact'),
    ('RAGU Score',        'ragu_score'),
    ('Amount Financed',   'amt_financed_x'),
    ('Weighted LTV',      'ltv'),
    ('Weighted APR',      'apr'),
]

EXCEL_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = 'barebones_ragu.xlsx'

sheet_name = EXCEL_SHEET_MAP[granularity]
sorted_vintages = sorted(all_df['vintage'].unique())

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1
all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())

for lob in all_export_lobs:
    lob_data = all_df[all_df.lob == lob].set_index('vintage')

    ws.cell(row=current_row, column=1, value=lob)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in lob_data.index:
                ws.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
        current_row += 1

    current_row += 1  # blank separator row

wb.save(EXCEL_OUTPUT)
print(f"Saved to {EXCEL_OUTPUT} (sheet: {sheet_name})")
print(f"  {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")

Saved to barebones_ragu.xlsx (sheet: Data Tables (W))
  9 groups x 14 periods


In [11]:
# =============================================================================
# CELL 10: CSV OUTPUT AND DIAGNOSTICS
# =============================================================================


run_sandbox = False
run_historical = False


col_rename = {
    'ms_original': 'model_score',
    'gross_loss_impact': 'gross_loss',
    'recovery_impact': 'recovery',
    'ltv_impact': 'ltv',
    'apr_impact': 'apr',
}
final_cols = ['lob', 'vintage', 'model_score', 'gross_loss', 'recovery', 'ltv', 'apr', 'ragu_score', 'amt_financed_x']
all_df['month_run'] = pd.Timestamp.now().strftime('%Y M%m')

# Select original names if available, otherwise use already-renamed names
if 'ms_original' in all_df.columns:
    source_cols = ['lob', 'vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed_x']
    output_df = all_df[source_cols + ['month_run']].rename(columns=col_rename)
else:
    output_df = all_df[final_cols + ['month_run']].copy()

output_df.to_csv('all_df.csv', index=False)
print(f"Saved all_df.csv ({len(output_df)} rows)")

display(output_df.drop(columns='month_run').head(20))

# --- Redshift Upload ---

def upload_ragu_to_redshift(df, table='sandbox.ragu_monthend_current'):
    """Upload a DataFrame to a Redshift table via INSERT INTO VALUES.
    Drops and recreates the table each run for a clean refresh."""
    upload_df = df.copy().reset_index(drop=True)
    upload_df['insert_column'] = (
        "('" + upload_df['lob'].astype(str)
        + "', '" + upload_df['vintage'].astype(str)
        + "', " + upload_df['model_score'].round(4).astype(str)
        + ", " + upload_df['gross_loss'].round(4).astype(str)
        + ", " + upload_df['recovery'].round(4).astype(str)
        + ", " + upload_df['ltv'].round(4).astype(str)
        + ", " + upload_df['apr'].round(4).astype(str)
        + ", " + upload_df['ragu_score'].round(4).astype(str)
        + ", '" + upload_df['month_run'].astype(str)
        + "')"
    )
    values_str = upload_df['insert_column'].str.cat(sep=',').replace("'nan'", 'null')

    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {table};")
        cur.execute(f"""
            CREATE TABLE {table} (
                lob         VARCHAR(10),
                vintage     VARCHAR(20),
                model_score FLOAT,
                gross_loss  FLOAT,
                recovery    FLOAT,
                ltv         FLOAT,
                apr         FLOAT,
                ragu_score  FLOAT,
                month_run   VARCHAR(10)
            );
        """)
        cur.execute(f"INSERT INTO {table} VALUES {values_str}")
        conn.commit()
    print(f"Uploaded {len(upload_df)} rows to {table}")


def upload_ragu_historical(month_run_val, table='sandbox.ragu_monthend',
                           source='sandbox.ragu_monthend_current'):
    """Append current-table rows into historical with current_version_flag.
    Uses temp-table DELETE + re-INSERT instead of UPDATE for Redshift performance.
    Idempotent: deletes any existing rows for this month_run before inserting."""
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {table} (
                lob                  VARCHAR(10),
                vintage              VARCHAR(20),
                model_score          FLOAT,
                gross_loss           FLOAT,
                recovery             FLOAT,
                ltv                  FLOAT,
                apr                  FLOAT,
                ragu_score           FLOAT,
                month_run            VARCHAR(10),
                current_version_flag SMALLINT DEFAULT 0
            );
        """)
        try:
            cur.execute(f"ALTER TABLE {table} ADD COLUMN current_version_flag SMALLINT DEFAULT 0")
        except Exception:
            pass

        cur.execute(f"DELETE FROM {table} WHERE month_run = '{month_run_val}'")

        cur.execute(f"CREATE TEMP TABLE _prev_current AS SELECT * FROM {table} WHERE current_version_flag = 1")
        cur.execute(f"DELETE FROM {table} WHERE current_version_flag = 1")
        cur.execute(f"""
            INSERT INTO {table}
            SELECT lob, vintage, model_score, gross_loss, recovery, ltv, apr, ragu_score, month_run, 0
            FROM _prev_current
        """)
        cur.execute("DROP TABLE _prev_current")

        cur.execute(f"INSERT INTO {table} SELECT *, 1 FROM {source}")
        conn.commit()
    print(f"Historical table {table} updated (month_run={month_run_val}, flag=1)")

if run_sandbox:
    month_run_val = output_df['month_run'].iloc[0]
    sandbox_df = output_df[output_df['vintage'] < month_run_val].drop(columns='amt_financed_x')
    print(f"Filtered to {len(sandbox_df)} rows (excluded vintages >= {month_run_val})")
    upload_ragu_to_redshift(sandbox_df)

if run_historical:
    upload_ragu_historical(output_df['month_run'].iloc[0])

Saved all_df.csv (126 rows)


,lob,vintage,model_score,gross_loss,recovery,ltv,apr,ragu_score,amt_financed_x
0,AN,2025-12-28/2026-01-03,140.489577,3.460489,4.353125,3.656362,0.970206,152.929760,3136288.37
1,AN,2026-01-04/2026-01-10,140.531838,3.573757,3.829143,3.538741,0.578127,152.051605,2865512.31
2,AN,2026-01-11/2026-01-17,140.455562,2.981565,3.075068,4.485800,0.601748,151.599743,2787608.89
3,AN,2026-01-18/2026-01-24,140.254720,3.646757,2.221830,3.694842,0.061543,149.879691,2478908.51
4,AN,2026-01-25/2026-01-31,140.423006,2.841823,2.259398,4.898943,1.089352,151.512522,3139821.65
5,AN,2026-02-01/2026-02-07,139.657795,2.958231,1.770772,4.111517,-0.097308,148.401007,3475913.53
6,AN,2026-02-08/2026-02-14,140.740186,3.003321,1.416120,4.172500,0.481342,149.813469,3939777.66
7,AN,2026-02-15/2026-02-21,140.266130,3.107341,0.943337,4.770535,0.803616,149.890959,5484726.75
8,AN,2026-02-22/2026-02-28,139.493092,2.304262,0.986003,5.203762,0.216189,148.203308,7535243.92
9,AN,2026-03-01/2026-03-07,141.278659,2.853977,2.180461,4.191118,0.671753,151.175969,5887487.43
